# Estimativa de Custo

Notebook responsável por estimar custo de avaliação de um determinado modelo em todo o dataset.

In [1]:
import pandas as pd
import numpy as np
from src.utils import length_function_tkt, carregar_dataset, gerar_hash_md5
from src.prompts.prompts_template import template

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate

In [2]:
df = carregar_dataset(filename='../data/train-00000-of-00001.parquet')
df.shape

Dataset lido com sucesso! shape: (3887, 3)


(3887, 3)

In [3]:
df = df[(df['data_category_QA']=='positivo') | (df['data_category_QA']=='negativo')]
df.shape

(3290, 3)

In [4]:
df['md5'] = df.content.apply(gerar_hash_md5)
df

,content,question,data_category_QA,md5
0,PERGUNTAS FREQUENTES\n\n ...,Roupas troca prazo?,positivo,1f98b127ca21922a8f960cc902a1ae8e
1,PERGUNTAS FREQUENTES\n\n ...,Qual email usar?,positivo,1f98b127ca21922a8f960cc902a1ae8e
2,Perguntas Frequentes\n\nQuais as formas de pag...,Quais serviços usados para realizar envios?,positivo,e435c0c2d64653c51c92cbe286264217
3,Perguntas Frequentes\n\nQuais as formas de pag...,Como ser notificado de disponibilidade novamen...,positivo,e435c0c2d64653c51c92cbe286264217
4,Perguntas Frequentes\n\n1. QUAIS SÃO AS FORMAS...,Quais tipos cartões aceitam para realizar paga...,positivo,d5f7d9465ecef75d8a7a04a2767fb0b0
...,...,...,...,...
3882,Perguntas frequentes Lojas 1. Qual é o horário...,"¿Quién abrió más tiendas en Europa primero, Ac...",negativo,da9a608666adba36a74f25a94cc31e0b
3883,Perguntas frequentes Lojas 1. Qual é o horário...,"¿Quién vende más, acción o Walmart?",negativo,da9a608666adba36a74f25a94cc31e0b
3884,Como posso enviar dinheiro pelo site westernun...,"¿Quién es mayor, Neymar o Messi?",negativo,8eea459e8b9d58e1dec447f8bba07406
3885,Como posso enviar dinheiro pelo site westernun...,"¿Qué banco creó UPI primero, ICICI o HDFC?",negativo,8eea459e8b9d58e1dec447f8bba07406


In [5]:
def transform_dataframe_in_documents(df):
    docs = []
    for _, row in df.iterrows():
        if row['data_category_QA'] in ['positivo', 'negativo']:
            doc = Document(
                    page_content = row['content'],
                    metadata = {'data_category_QA': row['data_category_QA'],
                                'md5': row['md5']
                            }
                )
            
            docs.append(doc)
            # break
    
    return docs

def gerar_chunks(documents):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 512,
        chunk_overlap = 100,
        length_function = length_function_tkt,
        separators = ["\n\n", "\n", ".", " ", ""]
    )

    # Gerando chunks e mantendo metadados do documento pai
    chunks = text_splitter.split_documents(documents)

    return chunks

In [6]:
documents = transform_dataframe_in_documents(df)
chunks = gerar_chunks(documents)

In [7]:
print(len(documents))
print(len(chunks))

3290
16408


In [8]:
tkt_contents = [length_function_tkt(chunk.page_content) for chunk in chunks]
len(tkt_contents)

16408

In [9]:
print(np.min(tkt_contents))
print(np.max(tkt_contents))
print(np.mean(tkt_contents))
print(np.median(tkt_contents))

1
512
376.7560336421258
443.0


In [10]:
mean_tkt_content = np.mean(tkt_contents)
mean_tkt_content

np.float64(376.7560336421258)

In [11]:
prompt_template = PromptTemplate.from_template(template=template)
prompt_template

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Você é um profissional especializado em Perguntas e Respostas.\nSeu OBJETIVO é verificar se é possível responder a pergunta APENAS com os dados contidos no contexto.\n\nResponda sempre em formato JSON válido. NÃO INSIRA nenhuma explicação ou conteúdo qualquer antes ou depois do json.\nGere APENAS UM json.\n\nCaso as informações do contexto NÃO SEJAM suficientes, incompletas ou não estejam presentes no contexto para responder a pergunta, responda negativo.\nExemplo de formato esperado:\n{{\n    "result": "negativo"\n}}\n\nCaso as informações do contexto ESTEJAM PRESENTES e SEJAM suficientes para responder corretamente a pergunta, responda positivo.\nExemplo de formato esperado:\n{{\n    "result": "positivo"\n}}\n\nSe você não souber a resposta, não tente inventar.\n\nContexto: {context}\n\nPergunta: {question}\n\nJSON de Saída:')

In [12]:
df['tkt_input'] = df.question.apply(lambda x: length_function_tkt(prompt_template.format(context='', question=x)) + (mean_tkt_content*3))

In [13]:
df

,content,question,data_category_QA,md5,tkt_input
0,PERGUNTAS FREQUENTES\n\n ...,Roupas troca prazo?,positivo,1f98b127ca21922a8f960cc902a1ae8e,1349.268101
1,PERGUNTAS FREQUENTES\n\n ...,Qual email usar?,positivo,1f98b127ca21922a8f960cc902a1ae8e,1345.268101
2,Perguntas Frequentes\n\nQuais as formas de pag...,Quais serviços usados para realizar envios?,positivo,e435c0c2d64653c51c92cbe286264217,1351.268101
3,Perguntas Frequentes\n\nQuais as formas de pag...,Como ser notificado de disponibilidade novamen...,positivo,e435c0c2d64653c51c92cbe286264217,1355.268101
4,Perguntas Frequentes\n\n1. QUAIS SÃO AS FORMAS...,Quais tipos cartões aceitam para realizar paga...,positivo,d5f7d9465ecef75d8a7a04a2767fb0b0,1355.268101
...,...,...,...,...,...
3882,Perguntas frequentes Lojas 1. Qual é o horário...,"¿Quién abrió más tiendas en Europa primero, Ac...",negativo,da9a608666adba36a74f25a94cc31e0b,1358.268101
3883,Perguntas frequentes Lojas 1. Qual é o horário...,"¿Quién vende más, acción o Walmart?",negativo,da9a608666adba36a74f25a94cc31e0b,1352.268101
3884,Como posso enviar dinheiro pelo site westernun...,"¿Quién es mayor, Neymar o Messi?",negativo,8eea459e8b9d58e1dec447f8bba07406,1352.268101
3885,Como posso enviar dinheiro pelo site westernun...,"¿Qué banco creó UPI primero, ICICI o HDFC?",negativo,8eea459e8b9d58e1dec447f8bba07406,1356.268101


In [14]:
tkt_output = length_function_tkt(' {\n    "result": "positivo"\n} ')
tkt_output

11

In [15]:
sum_tkt_output = tkt_output * df.shape[0]
sum_tkt_output

36190

In [16]:
sum_tkt_input = df.tkt_input.sum()
print(sum_tkt_input)

4465174.0520477835


In [17]:
# gpt-3.5-turbo-instruct

custo_dolar = (1.5 * sum_tkt_input / 1000000) + (2 * sum_tkt_output / 1000000)
print(custo_dolar)

6.770141078071675


In [18]:
# gpt-5-nano

custo_dolar = (0.05 * sum_tkt_input / 1000000) + (0.4 * sum_tkt_output / 1000000)
print(custo_dolar)

0.23773470260238916


In [19]:
# gpt-5.4-nano
# $0.20
# $0.02
# $1.25

custo_dolar = (0.2 * sum_tkt_input / 1000000) + (1.25 * sum_tkt_output / 1000000)
print(custo_dolar)

0.9382723104095567
